# EDA — Ames House Prices

Split across notebooks in this directory, one per step:

0. Load & orient
1. Target variable — `SalePrice`
2. Missingness
3. Univariate
4. Bivariate vs target
5. Multicollinearity
6. Outliers
7. Wrap-up

## 0. Load & orient

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# load csv data for both train and test
df_train = pd.read_csv("../../data/train.csv")
df_test = pd.read_csv("../../data/test.csv")

In [ ]:
df_train.shape, df_test.shape

In [ ]:
df_train.info()

In [ ]:
df_test.info()

In [ ]:
df_train.columns

In [ ]:
df_test.columns

In [ ]:
len(df_train.columns)

In [ ]:
df_train.head()

In [ ]:
df_train.tail()

In [ ]:
df_test.head()

In [ ]:
df_test.tail()

In [ ]:
# Feature type categorization, from data_description.txt
# each list is sorted() so the final order is alphabetical regardless of
# how the source below groups them (kept grouped here for readability)

IDENTIFIER = ["Id"]
TARGET = ["SalePrice"]

NUMERIC_COLS = sorted([
    "LotFrontage", "LotArea", "MasVnrArea", "BsmtFinSF1", "BsmtFinSF2",
    "BsmtUnfSF", "TotalBsmtSF", "1stFlrSF", "2ndFlrSF", "LowQualFinSF",
    "GrLivArea", "BsmtFullBath", "BsmtHalfBath", "FullBath", "HalfBath",
    "BedroomAbvGr", "KitchenAbvGr", "TotRmsAbvGrd", "Fireplaces",
    "GarageCars", "GarageArea", "WoodDeckSF", "OpenPorchSF",
    "EnclosedPorch", "3SsnPorch", "ScreenPorch", "PoolArea", "MiscVal",
    # temporal — usually engineered into ages/deltas rather than used raw
    "YearBuilt", "YearRemodAdd", "GarageYrBlt", "MoSold", "YrSold",
])

# has a genuine order -> integer-encode in that order, don't one-hot
ORDINAL_COLS = sorted([
    "OverallQual", "OverallCond", "LotShape", "LandSlope",
    "ExterQual", "ExterCond", "BsmtQual", "BsmtCond", "BsmtExposure",
    "BsmtFinType1", "BsmtFinType2", "HeatingQC", "KitchenQual",
    "Functional", "FireplaceQu", "GarageFinish", "GarageQual",
    "GarageCond", "PavedDrive", "PoolQC", "Utilities",
])

# no inherent order -> one-hot/dummy encode
NOMINAL_COLS = sorted([
    "MSSubClass",  # gotcha: stored as int, but it's a dwelling-type code
    "MSZoning", "Street", "Alley", "LandContour", "LotConfig",
    "Neighborhood", "Condition1", "Condition2", "BldgType", "HouseStyle",
    "RoofStyle", "RoofMatl", "Exterior1st", "Exterior2nd", "MasVnrType",
    "Foundation", "Heating", "CentralAir", "Electrical", "GarageType",
    "MiscFeature", "SaleType", "SaleCondition", "Fence",
])

all_cols = NUMERIC_COLS + ORDINAL_COLS + NOMINAL_COLS
expected = set(df_train.columns) - set(IDENTIFIER) - set(TARGET)
assert set(all_cols) == expected, set(all_cols) ^ expected
assert len(all_cols) == len(expected)  # no duplicates
len(NUMERIC_COLS), len(ORDINAL_COLS), len(NOMINAL_COLS)

In [ ]:
# check column set for train and test set
expected = set(df_test.columns) ^ (set(df_train.columns) - {"SalePrice"})
assert not expected, expected
expected, len(expected)

In [ ]:
# check duplicate rows: train dataframe
df_train.drop(columns="Id").duplicated().sum()

In [ ]:
# check duplicate rows: test df
df_test.drop(columns="Id").duplicated().sum()

In [ ]:
# Id uniqueness: train df
df_train["Id"].is_unique

In [ ]:
# Id uniqueness: test df
df_test["Id"].is_unique

In [ ]:
# dtype consistency: train vs test, for shared columns
dtypes = pd.DataFrame({"train": df_train.dtypes, "test": df_test.dtypes})
dtypes[dtypes["train"] != dtypes["test"]]